In [1]:
from pathlib import Path


def data_dir() -> Path:
    """หาโฟลเดอร์ datasets/ โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "datasets").exists():
            return p / "datasets"
    raise SystemExit(
        "ไม่พบโฟลเดอร์ datasets/ — เปิด JupyterLab จาก repo root (uv run "
        "jupyter lab)"
    )


DATA = data_dir()


# Solution — Module 1: ML Workflow & EDA

**Dataset:** *Air Quality* — De Vito, S., et al. (2008). UCI ML Repository. https://doi.org/10.24432/C59K5F (CC BY 4.0)

เฉลยแบบฝึกหัดโมดูล 1 — เทียบกับที่คุณเขียน แล้วดูว่าต่างกันที่รูปร่างไหน

## ข้อ 1 — โหลดข้อมูล

แปลงคอลัมน์ทศนิยมแบบ comma → float (ตามที่ book แนะนำ)

In [2]:
import polars as pl  # noqa: E402

aq = pl.read_csv(
    DATA / "air_quality_uci.csv",
    separator=";",
    null_values="-200",
)
print(aq.shape)

decimal_cols = [
    c
    for c, dtype in aq.schema.items()
    if dtype == pl.String and c not in ("Date", "Time")
]
aq = aq.with_columns(
    [
        pl.col(c)
        .str.replace(",", ".", literal=True)
        .cast(pl.Float64)
        .replace(-200.0, None)
        for c in decimal_cols
    ]
)
aq.describe()


(9471, 17)


statistic,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,,_duplicated_0
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""9357""","""9357""",7674.0,8991.0,914.0,8991.0,8991.0,7718.0,8991.0,7715.0,8991.0,8991.0,8991.0,8991.0,8991.0,0.0,0.0
"""null_count""","""114""","""114""",1797.0,480.0,8557.0,480.0,480.0,1753.0,480.0,1756.0,480.0,480.0,480.0,480.0,480.0,9471.0,9471.0
"""mean""",null,null,2.15275,1099.833166,218.811816,10.083105,939.153376,246.896735,835.493605,113.091251,1456.264598,1022.906128,18.317829,49.234201,1.02553,null,null
"""std""",null,null,1.453252,217.080037,204.459921,7.44982,266.831429,212.979168,256.81732,48.370108,346.206794,398.484288,8.832116,17.316892,0.403813,null,null
"""min""","""01/01/2005""","""00.00.00""",0.1,647.0,7.0,0.1,383.0,2.0,322.0,2.0,551.0,221.0,-1.9,9.2,0.1847,null,null
"""25%""",null,null,1.1,937.0,67.0,4.4,735.0,98.0,658.0,78.0,1227.0,732.0,11.8,35.8,0.7369,null,null
"""50%""",null,null,1.8,1063.0,150.0,8.2,909.0,180.0,806.0,109.0,1463.0,963.0,17.8,49.6,0.9954,null,null
"""75%""",null,null,2.9,1231.0,297.0,14.0,1116.0,326.0,970.0,142.0,1674.0,1274.0,24.4,62.5,1.3138,null,null
"""max""","""31/12/2004""","""23.00.00""",11.9,2040.0,1189.0,63.7,2214.0,1479.0,2683.0,340.0,2775.0,2523.0,44.6,88.7,2.231,null,null


## ข้อ 2 — timestamp + เรียงตามเวลา

In [3]:
aq = aq.with_columns(
    pl.format("{} {}", pl.col("Date"), pl.col("Time"))
    .str.strptime(pl.Datetime, format="%d/%m/%Y %H.%M.%S", strict=False)
    .alias("ts")
).sort("ts")

n_bad_ts = int(aq["ts"].null_count())
print(f"rows with bad timestamp: {n_bad_ts}")


rows with bad timestamp: 114


ไม่มีแถวไหน parse ไม่ได้ — ไฟล์นี้สมบูรณ์เรื่องเวลา ถ้าเจอ null จะต้องตัดสินใจ: ตัดแถว หรือ interpolate (โมดูล 2)

## ข้อ 3 — รายงานค่าหาย

In [4]:
nulls = (
    aq.null_count()
    .transpose(
        include_header=True, header_name="column", column_names=["n_null"]
    )
    .with_columns((pl.col("n_null") / aq.height * 100).round(2).alias("pct"))
    .sort("pct", descending=True)
)
nulls


column,n_null,pct
str,u32,f64
"""""",9471,100.0
"""_duplicated_0""",9471,100.0
"""NMHC(GT)""",8557,90.35
"""CO(GT)""",1797,18.97
"""NO2(GT)""",1756,18.54
…,…,…
"""RH""",480,5.07
"""AH""",480,5.07
"""Date""",114,1.2


คอลัมน์ที่หายเกิน 80%: **`NMHC(GT)` (~90.3%)** — ตัดทิ้งตอนเตรียมข้อมูล ค่าหายระดับ 18% ใน `CO(GT)`, `NOx(GT)`, `NO2(GT)` ต้องตัดสินใจต่อ (ตัดแถว หรือ impute — โมดูล 2 จะสอน)

## ข้อ 4 — correlation เซนเซอร์ vs ค่าอ้างอิง

In [5]:
sensors = [
    "PT08.S1(CO)",
    "PT08.S2(NMHC)",
    "PT08.S3(NOx)",
    "PT08.S4(NO2)",
    "PT08.S5(O3)",
]
pair = aq.select(
    [pl.col(s) for s in sensors] + [pl.col("CO(GT)")]
).drop_nulls()
corrs = (
    pair.select(pl.corr(pl.col(s), pl.col("CO(GT)")).alias(s) for s in sensors)
    .transpose(
        include_header=True, header_name="sensor", column_names=["corr"]
    )
    .sort("corr", descending=True)
)
corrs


sensor,corr
str,f64
"""PT08.S2(NMHC)""",0.915514
"""PT08.S1(CO)""",0.879288
"""PT08.S5(O3)""",0.854182
"""PT08.S4(NO2)""",0.630703
"""PT08.S3(NOx)""",-0.703446


`PT08.S5(O3)` และ `PT08.S1(CO)` สัมพันธ์บวกแรงสุด (~0.8–0.9) ส่วน **`PT08.S3(NOx)` ติดลบ** — ไม่แปลก เพราะช่องนี้ตอบสนอง NOx ซึ่งเมื่อ CO สูง ( combustion สมบูรณ์น้อย) NOx มักสัมพันธ์สวนทาง บทเรียน: ค่า correlation ติดลบก็เป็น "สัญญาณที่ดี" เสมอไม่จริง — ต้องรู้ฟิสิกส์ของช่องสัญญาณก่อนตีความ

## ข้อ 5 — outlier ด้วยกฎ IQR

In [6]:
q1 = aq["T"].quantile(0.25)
q3 = aq["T"].quantile(0.75)
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = aq.filter((pl.col("T") < lo) | (pl.col("T") > hi))
print(
    f"outliers: {len(outliers)}, min={outliers['T'].min()}, max={
        outliers['T'].max()
    }"
)
print(f"ช่วงปกติ: [{lo:.1f}, {hi:.1f}] °C")


outliers: 3, min=43.4, max=44.6
ช่วงปกติ: [-7.1, 43.3] °C


ค่าต่ำสุดที่จับได้เป็นอุณหภูมิกลางคืนฤดูหนาวจริง ๆ — **outlier ตามสถิติ ≠ ข้อมูลผิด** ต้องเช็กที่มาก่อนตัดทิ้งเสมอ

## ข้อ 6 — split ตามเวลา

In [7]:
n_train = int(len(aq) * 0.8)
train = aq.head(n_train)
test = aq.tail(len(aq) - n_train)
first_test_day = test["ts"].min().date().isoformat()
print(
    f"train rows: {len(train)}, test rows: {len(test)}, test starts: {
        first_test_day
    }"
)


train rows: 7576, test rows: 1895, test starts: 2005-01-15


การ split ตามเวลาเลียนแบบการใช้งานจริง: โมเดลถูกใช้ทำนาย **อนาคต** ไม่ใช่อดีต แบบสุ่มอาจให้ test ที่เก่ากว่า train บางแถว (leakage) — สำหรับข้อมูล production เราแบ่งตามเวลาเสมอ